# Modelado Avanzado de Dispositivo Termopar: Ajuste de Barrera Schottky Back-to-Back

Este Notebook aborda la caracterización eléctrica de un termopar a partir de datos experimentales de corriente y voltaje. Inicialmente, se intentó un ajuste simple para encontrar el coeficiente de Seebeck, pero la naturaleza no lineal de los datos sugirió la presencia de contactos no óhmicos.

Por ello, se implementó un modelo avanzado basado en **dobles barreras Schottky (Back-to-Back)** para modelar con precisión el comportamiento asimétrico del dispositivo bajo gradiente térmico.

## Dependencias

In [ ]:
!pip install lmfit

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lmfit as lm
from google.colab import drive

## 1. Carga y Visualización Preliminar de Datos
Se cargan los datos crudos obtenidos del instrumento de medición (Source Measure Unit - SMU) para inspeccionar su comportamiento general, comparando las curvas en frío y con un gradiente de calor aplicado.

In [ ]:
try:
    drive.mount('/content/drive', force_remount=True)
    df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Datos_Termopar.csv", header=None)
    df.columns = ['Voltaje', 'Corriente en Frío', 'Corriente en calor']
    print("Datos cargados correctamente.\n")
    
    # Visualización inicial estilo Origin
    plt.figure(figsize=(10, 6))
    color_frio = '#000000' 
    color_calor = '#FF0000' 

    plt.scatter(df['Voltaje'], df['Corriente en Frío'], label='Corriente en Frío',
                marker='o', s=40, edgecolors='black', alpha=0.8, color=color_frio)
    plt.scatter(df['Voltaje'], df['Corriente en calor'], label='Corriente en Calor',
                marker='x', s=40, edgecolors='black', alpha=0.8, color=color_calor)

    plt.xlabel('Voltaje (V)', fontsize=14, fontweight='bold')
    plt.ylabel('Corriente (A)', fontsize=14, fontweight='bold')
    plt.title('Curvas I-V del Termopar: Frío vs. Calor', fontsize=16, fontweight='bold')
    plt.legend(fontsize=12, frameon=True, shadow=True, borderpad=1)
    plt.grid(True, which='major', linestyle='--', alpha=0.7, color='#A9A9A9')
    plt.minorticks_on()
    plt.grid(True, which='minor', linestyle=':', alpha=0.5, color='#D3D3D3')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Error al cargar los datos: {e}")

## 2. Ajuste Computacional: Modelo de Doble Barrera Schottky
Como se observa en la gráfica anterior, el dispositivo no exhibe un comportamiento óhmico puro. Para realizar un ajuste termodinámico más realista, extraemos los datos bajo estrés térmico y aplicamos una regresión no lineal utilizando la librería `lmfit`.

El modelo asume dos barreras Schottky en oposición (back-to-back), calculando las alturas de barrera ($\phi_{b1}, \phi_{b2}$) y los factores de idealidad ($n_1, n_2$).

In [ ]:
if 'df' in locals():
    # Extracción de variables
    V_datos = df['Voltaje'].values
    I_datos_reales = df['Corriente en calor'].values
    I_datos_para_fit = -I_datos_reales # Inversión requerida por el modelo matemático

    # Constantes físicas y del material (Ajustadas para MoS2 u homólogos)
    s1 = 100e-8 # cm^-2
    s2 = 15e-8  # cm^-2
    T = 300     # K (Temperatura base del análisis)
    A = 80.3    # Constante de Richardson, A cm^-2 K^-2
    k = .00008617 # Constante de Boltzmann eV K^-1
    q = 1

    # Inicialización de Parámetros
    pm = lm.Parameters()
    pm.add('phib01', value=0.5, vary=True, min=0, max=2)
    pm.add('phib02', value=0.5, vary=True, min=0, max=2)
    pm.add('n1', value=1, vary=True, min=0.1, max=5)
    pm.add('n2', value=1, vary=True, min=0.1, max=5)

    # Definición de la Ecuación Gobernante (Back-to-Back Schottky)
    def func(paramsin, V):
        phib01 = paramsin['phib01'].value
        phib02 = paramsin['phib02'].value
        n1 = paramsin['n1'].value
        n2 = paramsin['n2'].value
        
        phib1 = phib01 + (q*V/2*(1-(1/n1)))
        phib2 = phib02 - (q*V/2*(1-(1/n2)))
        Is1 = s1 * A * T**2 * np.exp(-phib1/(k*T))
        Is2 = s2 * A * T**2 * np.exp(-phib2/(k*T))
        
        Itot = ((2*Is1*Is2*np.sinh((q*V)/(2*k*T))) / 
                ((Is1*np.exp((q*V)/(2*k*T))) + (Is2*np.exp((-q*V)/(2*k*T)))))
        return Itot

    # Optimización del modelo minimizando el error residual
    funcerr = lambda p, x, y: func(p, x) - y
    fitout = lm.minimize(funcerr, pm, args=(V_datos, I_datos_para_fit))
    fitted = fitout.params

    pars = [fitted['phib01'].value, fitted['phib02'].value, 
            fitted['n1'].value, fitted['n2'].value]

    # Graficación del ajuste vs datos experimentales
    plt.figure(figsize=(10, 6))
    plt.plot(V_datos, I_datos_reales, 'bo', label='Datos Experimentales (SMU)', markersize=5, alpha=0.6)
    I_linea_ajustada = func(fitted, V_datos)
    plt.plot(V_datos, -I_linea_ajustada, 'r-', linewidth=2, label='Modelo Ajustado (Back-to-Back)')

    # Ajuste de escala Y para visualización óptima
    padding = (np.max(I_datos_reales) - np.min(I_datos_reales)) * 0.10
    plt.ylim(np.min(I_datos_reales) - padding, np.max(I_datos_reales) + padding)

    # Caja de resultados
    texto_anotacion = rf"""$\bf{{Resultados\ de\ Optimización:}}$
$\beta_{{b01}} (Barrera\ 1) = {pars[0]:.3f}$ eV
$\beta_{{b02}} (Barrera\ 2) = {pars[1]:.3f}$ eV
$n_1 (Idealidad\ 1) = {pars[2]:.3f}$
$n_2 (Idealidad\ 2) = {pars[3]:.3f}$"""

    plt.text(0.05, 0.95, texto_anotacion, transform=plt.gca().transAxes, fontsize=11,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

    plt.xlabel('Voltaje (V)', fontsize=12)
    plt.ylabel('Corriente (A)', fontsize=12)
    plt.title('Ajuste de Doble Barrera Schottky a Datos Termoeléctricos', fontsize=14)
    plt.legend(loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()
else:
    print("No se pueden ejecutar los cálculos debido a un fallo en la carga de datos.")

## 3. Discusión y Análisis Crítico de Resultados

Aunque el modelo de doble barrera Schottky logra capturar la asimetría general de la curva I-V que el modelo original de estimación directa del coeficiente de Seebeck no pudo, se observa un **desajuste notable entre la curva teórica y los datos experimentales en la región de voltaje positivo**.

Este comportamiento subóptimo del ajuste se atribuye principalmente a dos fuentes de error experimental sistémico identificadas durante la toma de datos:

1.  **Incertidumbre Térmica (Fallo de Laboratorio):** El modelo asume un perfil térmico constante e ideal ($T = 300 K$ como base y gradientes perfectos). Sin embargo, fallos en la monitorización de temperatura del laboratorio impidieron capturar las fluctuaciones térmicas reales en las uniones. Esta carencia de datos de temperatura precisos y dinámicos obliga al modelo a trabajar con valores estáticos que no representan el fenómeno físico que ocurrió en tiempo real, diluyendo la extracción precisa de los parámetros termoeléctricos.
2.  **Descalibración del Source Measure Unit (SMU):** La inspección de los datos crudos en la primera gráfica revela una saturación anómala y "ruido estructural" escalonado en la lectura de corriente. Esto es un indicio claro de que el equipo de medición (SMU) se encontraba operando fuera de sus tolerancias de calibración, introduciendo artefactos en la medición de corriente que ningún modelo físico ideal puede replicar matemáticamente sin sobreajuste (overfitting).

**Conclusión:** La complejidad del ajuste matemático demuestra la presencia de contactos no óhmicos en el termopar. No obstante, para extraer un coeficiente de Seebeck altamente confiable, es mandatorio repetir el experimento garantizando una calibración estricta del SMU y una adquisición de datos térmicos sincronizada y precisa.